In [ ]:
import pandas as pd
import numpy as np
import requests
from datetime import datetime
from itertools import combinations
from scipy.spatial.distance import euclidean
from scipy.stats import zscore
from tslearn.piecewise import SymbolicAggregateApproximation
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
from tslearn.metrics import dtw
from stumpy import mpdist
from tqdm.notebook import tqdm  # for visible Jupyter progress bar

# --- Step 1: Load Metadata and Sample 50 Stations ---
metadata_path = "metadata_without_res_20250603.csv"
metadata_df = pd.read_csv(metadata_path, dtype=str)
selected_metadata = metadata_df.dropna(subset=["STATION_ID"]).sample(n=50, random_state=42)
station_ids = selected_metadata["STATION_ID"].unique().tolist()

# --- Step 2: Download Station Data ---
def download_station_data(station_id):
    today = datetime.today().strftime("%Y-%m-%d")
    url = f"https://www.waterrights.utah.gov/dvrtdb/daily-chart.asp?station_id={station_id}&end_date={today}&f=json"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200 and "data" in r.json():
            df = pd.DataFrame(r.json()["data"], columns=["date", "value"])
            df["date"] = pd.to_datetime(df["date"])
            df["value"] = pd.to_numeric(df["value"], errors="coerce")
            df = df.drop_duplicates(subset="date").set_index("date")
            return station_id, df.rename(columns={"value": station_id})
    except:
        pass
    return station_id, None

data_dict = {}
for sid in tqdm(station_ids, desc="📥 Downloading data"):
    sid, df = download_station_data(sid)
    if df is not None:
        data_dict[sid] = df

merged_df = pd.concat(data_dict.values(), axis=1)

# --- Step 3: Lookup Tables ---
station_names = dict(zip(selected_metadata["STATION_ID"], selected_metadata["MasterStationName"]))
lat_lookup = dict(zip(selected_metadata["STATION_ID"], selected_metadata["LAT"]))
lon_lookup = dict(zip(selected_metadata["STATION_ID"], selected_metadata["LON"]))

# --- Step 4: Pairwise Similarity Metrics ---
results = []
station_pairs = list(combinations(data_dict.keys(), 2))

for s1, s2 in tqdm(station_pairs, desc="🔄 Processing pairs"):
    try:
        valid = merged_df[[s1, s2]].dropna()
        v1 = valid[s1].values
        v2 = valid[s2].values
    except KeyError:
        continue

    z_norm_euclidean = sax_distance = dtw_distance = mp_dist_value = np.nan
    overlap_count = len(valid)

    if len(v1) >= 60 and np.std(v1) != 0 and np.std(v2) != 0:
        try:
            z1 = zscore(v1)
            z2 = zscore(v2)
            z_norm_euclidean = euclidean(z1, z2)
        except: pass

        try:
            series = np.vstack([v1, v2]).astype("float32")
            scaled = TimeSeriesScalerMeanVariance().fit_transform(series)
            sax_model = SymbolicAggregateApproximation(n_segments=30, alphabet_size_avg=5)
            sax_trans = sax_model.fit_transform(scaled)
            sax_distance = np.sum(sax_trans[0] != sax_trans[1])
        except: pass

        try:
            dtw_distance = dtw(v1, v2)
        except: pass

        try:
            mpdist_result = mpdist(v1, v2, 30)
            if mpdist_result.size > 0 and not np.isnan(mpdist_result).all():
                mp_dist_value = np.nanmax(mpdist_result)
        except: pass

    results.append({
        "Station1": s1,
        "Station2": s2,
        "Station1_Name": station_names.get(s1, ""),
        "Station2_Name": station_names.get(s2, ""),
        "Station1_LAT": lat_lookup.get(s1, ""),
        "Station1_LON": lon_lookup.get(s1, ""),
        "Station2_LAT": lat_lookup.get(s2, ""),
        "Station2_LON": lon_lookup.get(s2, ""),
        "Z_Norm_Euclidean": z_norm_euclidean,
        "SAX_Distance": sax_distance,
        "DTW_Distance": dtw_distance,
        "MPDist": mp_dist_value,
        "OverlapDays": overlap_count,
        "StationPage": f"https://waterrights.utah.gov/dvrtdb/daily-chart.asp?STATION_ID={s1},{s2}"
    })

# --- Step 5: Filter and Save with OR logic ---
results_df = pd.DataFrame(results)
cols_to_filter = ["Z_Norm_Euclidean", "SAX_Distance", "DTW_Distance", "MPDist"]
results_df[cols_to_filter] = results_df[cols_to_filter].apply(pd.to_numeric, errors="coerce")

filtered_df = results_df[
    ((results_df["Z_Norm_Euclidean"] >= 0) & (results_df["Z_Norm_Euclidean"] <= 10)) |
    ((results_df["SAX_Distance"] >= 0) & (results_df["SAX_Distance"] <= 10)) |
    ((results_df["DTW_Distance"] >= 0) & (results_df["DTW_Distance"] <= 50)) |
    ((results_df["MPDist"] >= 0) & (results_df["MPDist"] <= 0.3))
]

filtered_df.to_csv("jupyter_filtered_similarity_50_stations.csv", index=False)
print("✅ Filtered results saved: jupyter_filtered_similarity_50_stations.csv")